# 4.2 形状与下落速度模型

**学习目标**
- 理解雨滴轴比随直径变化的规律
- 掌握终端下落速度的经验公式
- 观察形状和速度对雷达变量的影响
- 比较不同水凝物类型的形状速度关系

**参考文献**：Ryzhkov & Zrnic, Chapter 4, pp. 396-430

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### 雨滴轴比模型

小型雨滴接近球形，大型雨滴因空气动力压扁呈椭球形：

$$b/a = 1 - 0.05D \quad (D \text{ in mm})$$

其中 $b/a$ 是短轴/长轴比，$D$ 是等效体积直径。

### 终端速度模型 (Atlas-Livesey)

$$V(D) = V_0 \left(1 - \exp\left(-\frac{D}{D_0}\right)\right)$$

对于雨滴，$V_0 \approx 9.65$ m/s，$D_0 \approx 0.9$ mm。

In [ ]:
def raindrop_axis_ratio(D):
    """
    计算雨滴轴比（直径单位mm）
    基于经验公式
    """
    D = np.asarray(D)
    # 球形小滴
    ar = np.where(D < 0.5, 1.0, 1 - 0.05 * D)
    # 大滴限制轴比下限
    ar = np.maximum(ar, 0.6)
    return ar

def terminal_velocity(D, hydrometeor_type='rain'):
    """
    计算终端下落速度
    D: 直径 (mm)
    hydrometeor_type: 'rain', 'snow', 'hail'
    """
    D = np.asarray(D)
    
    if hydrometeor_type == 'rain':
        # Atlas-Livesey 公式
        V0 = 9.65  # m/s
        D0 = 0.9   # mm
        V = V0 * (1 - np.exp(-(D / D0)**1.147))
    elif hydrometeor_type == 'snow':
        # 雪花（较慢）
        V = 1.3 * D**0.31  # m/s，简化公式
    elif hydrometeor_type == 'hail':
        # 冰雹（较快）
        V = np.sqrt(2 * 9.81 * D * 1e-3) * 0.8  # 简化重力公式
    else:
        V = np.zeros_like(D)
    
    return V

def calculate_fall_time(D, V_avg=None):
    """计算下落时间（简化）"""
    if V_avg is None:
        V_avg = terminal_velocity(D, 'rain')
    return D / (V_avg + 1e-10)  # 简化: 距离/速度

## 2. 形状与速度可视化

In [ ]:
def plot_shape_velocity(D_max=8.0, hydrometeor='rain'):
    """可视化形状和速度模型"""
    
    # 直径范围
    D = np.linspace(0.1, D_max, 200)
    
    # 轴比
    ar = raindrop_axis_ratio(D)
    
    # 终端速度
    V = terminal_velocity(D, hydrometeor)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 子图1: 轴比 vs 直径
    ax1 = axes[0, 0]
    ax1.plot(D, ar, 'b-', linewidth=2)
    ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='球形 (b/a=1)')
    ax1.axhline(y=0.6, color='red', linestyle=':', alpha=0.5, label='最小轴比 (0.6)')
    ax1.set_xlabel('直径 D (mm)', fontsize=11)
    ax1.set_ylabel('轴比 b/a', fontsize=11)
    ax1.set_title('雨滴轴比随直径的变化', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, D_max)
    ax1.set_ylim(0.5, 1.05)
    
    # 子图2: 终端速度 vs 直径
    ax2 = axes[0, 1]
    for htype, color in [('rain', 'blue'), ('snow', 'green'), ('hail', 'red')]:
        V_type = terminal_velocity(D, htype)
        ax2.plot(D, V_type, color=color, linewidth=2, label=htype.capitalize())
    ax2.set_xlabel('直径 D (mm)', fontsize=11)
    ax2.set_ylabel('终端速度 V (m/s)', fontsize=11)
    ax2.set_title('不同水凝物类型的终端速度', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, D_max)
    
    # 子图3: 雨滴形状示意图
    ax3 = axes[1, 0]
    D_samples = [1, 3, 5, 7]
    for i, D_val in enumerate(D_samples):
        ar_val = raindrop_axis_ratio(D_val)
        a = D_val / 2
        b = a * ar_val
        ellipse = plt.matplotlib.patches.Ellipse((i*2+1, 1), 2*a, 2*b, 
                                                  fill=False, color='blue', linewidth=2)
        ax3.add_patch(ellipse)
        ax3.text(i*2+1, -0.3, f'D={D_val}mm b/a={ar_val:.2f}', ha='center', fontsize=9)
        ax3.plot([i*2+1, i*2+1], [-b, b], 'b--', linewidth=1)
    ax3.set_xlim(-1, 9)
    ax3.set_ylim(-3, 3)
    ax3.set_aspect('equal')
    ax3.axis('off')
    ax3.set_title('雨滴形状随直径变化示意', fontsize=12)
    
    # 子图4: 速度-直径乘积（对降水通量的贡献）
    ax4 = axes[1, 1]
    N_D = 8000 * np.exp(-4.1 * D * (10**(-0.21)))  # MP分布简化
    mass_flux_rain = terminal_velocity(D, 'rain') * D**3 * N_D
    mass_flux_snow = terminal_velocity(D, 'snow') * D**3 * N_D
    ax4.semilogy(D, mass_flux_rain / mass_flux_rain.max(), 'b-', linewidth=2, label='雨滴')
    ax4.semilogy(D, mass_flux_snow / mass_flux_snow.max(), 'g--', linewidth=2, label='雪花')
    ax4.set_xlabel('直径 D (mm)', fontsize=11)
    ax4.set_ylabel('归一化质量通量', fontsize=11)
    ax4.set_title('质量通量对直径的依赖', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim(0, D_max)
    
    plt.tight_layout()
    plt.show()
    
    # 打印关键值
    print(f"\n=== 形状和速度关键值 ===")
    print(f"{'D (mm)':<10} {'轴比 b/a':<12} {'V_rain (m/s)':<15} {'V_snow (m/s)':<15}")
    print("-" * 52)
    for D_val in [1, 2, 3, 4, 5, 6, 8]:
        print(f"{D_val:<10.1f} {raindrop_axis_ratio(D_val):<12.3f} {terminal_velocity(D_val, 'rain'):<15.2f} {terminal_velocity(D_val, 'snow'):<15.2f}")

In [ ]:
interact_sv = interact(plot_shape_velocity,
    D_max=widgets.FloatSlider(min=2, max=10, step=0.5, value=8.0, 
                             description='最大直径 (mm)'),
    hydrometeor=widgets.Dropdown(options=[('雨滴', 'rain'), ('雪花', 'snow'), ('冰雹', 'hail')],
                               value='rain', description='水凝物类型')
)

## 3. 练习题

1. **轴比变化**：直径5mm的雨滴轴比约是多少？这比1mm的雨滴有何不同？

2. **终端速度**：直径3mm的雨滴和雪花的终端速度分别为多少？为何差异很大？

3. **物理机制**：为什么大型雨滴会是扁平的而不是球形？

4. **速度饱和**：当雨滴直径很大时（>5mm），终端速度是否会持续增加？为什么？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 4, Springer.
- Atlas, D., and C.W.U. Ulbrich, 1974: The physical basis for attenuation-rainfall relationships. *Preprints*
- Brandes, E.A., et al., 2002: Relationships between reflectivity, rain rate, and drop size. *J. Appl. Meteor.*